# Notebook 1 — Exploring a Hydroacoustic Recording

**Listening to the Ocean | Unit 3: Signal Processing Basics**

In this self-guided activity, we will explore a real **blue whale recording from Cascadia Basin** using Python. No previous programming or signal-processing experience is required: read each explanation, then run the code cell below it using **Shift + Enter**.

We will:

1. load and listen to an underwater recording;
2. inspect its waveform (amplitude over time);
3. identify prominent frequencies using a Fast Fourier Transform (FFT);
4. create and interpret a spectrogram;
5. apply a band-pass filter and compare the result with the original;
6. use the figures to explain what the recording does and does not tell us.

### Important — the supplied audio has been sped up 10 times

Ocean Networks Canada (ONC) accelerated this blue whale audio **10×** so its low-frequency calls are easier to hear. **Our code analyses the supplied, sped-up MP3 — not the original-speed recording.**

- **Playback frequency = original frequency × 10.** A peak at 150 Hz in the supplied file corresponds to approximately 15 Hz in the original sound.
- **Playback time = original time ÷ 10.** A 6-second interval in the supplied file represents approximately 60 seconds in the original recording.
- Throughout this notebook, figure axes are labelled **playback time** and **playback frequency**. When estimating original underwater values, we will apply the factor of 10 explicitly.
- The MP3 is **not a calibrated pressure recording**. Amplitudes are relative digital values, **not pascals or dB re 1 µPa**. MP3 compression may also affect detailed quantitative analysis.

**Recording:** *Blue Whale in Cascadia Basin*, Ocean Networks Canada; recorded 18 October 2013 at approximately 2,662 m depth. Original source and playback details:

https://ibase.oceannetworks.ca/view-item?i=9838

## Before you begin — files, libraries and running code

Save this notebook (`.ipynb`) and **`Blue Whale in Cascadia Basin (wav).mp3`** in the **same folder**. The filename must match exactly.

We will use:

- **NumPy** — calculations and the FFT;
- **Matplotlib** — graphs;
- **SciPy** — spectrograms and digital filters;
- **librosa** — reading the MP3 recording;
- **IPython** — an audio player inside the notebook.

If the packages are missing, open **Command Prompt** on Windows or **Terminal** on macOS and run the appropriate command:

**Windows:**

```bash
py -m pip install notebook numpy matplotlib scipy librosa soundfile
```

If `py` is not recognised on Windows, try `python` instead. **macOS:**

```bash
python3 -m pip install notebook numpy matplotlib scipy librosa soundfile
```

Restart the notebook kernel after installing libraries and run the cells from the top.

**How to run the notebook:** A *Markdown cell* contains these explanations; a *code cell* contains Python instructions. Click a code cell and press **Shift + Enter**. Outputs and plots appear beneath the cell.

**Troubleshooting:** `ModuleNotFoundError` means a library is not available in the notebook's Python environment. `FileNotFoundError` usually means the MP3 is not in the notebook's working folder or its filename differs. If a later cell says that a variable is undefined, run the previous cells in order.

In [ ]:
# Import the Python libraries used throughout this notebook.
import numpy as np                        # Calculations and the FFT
import matplotlib.pyplot as plt           # Graphs
import librosa                            # Load the MP3 audio
from pathlib import Path                 # Check file locations
from IPython.display import Audio, display  # Audio player in Jupyter
from scipy.signal import spectrogram, butter, sosfiltfilt

In [ ]:
# The MP3 must be stored in the same folder as this notebook.
file_path = Path("Blue Whale in Cascadia Basin (wav).mp3")

if not file_path.is_file():
    raise FileNotFoundError(
        f"Cannot find '{file_path.name}'. Place the MP3 in the notebook's "
        "working folder, check the exact filename, and rerun this cell."
    )

# sr=None preserves the MP3 playback sampling frequency.
# mono=True converts multiple audio channels into a single signal if needed.
# These settings do NOT undo the 10× speed-up applied by ONC.
audio, fs = librosa.load(str(file_path), sr=None, mono=True)

print("Loaded:", file_path.name)
print(f"Playback sampling frequency: {fs:,} samples/s")
print(f"Number of audio samples: {len(audio):,}")

In [ ]:
# Centre the signal around zero by subtracting its mean.
# We keep these samples in their original relative digital units.
audio = audio - np.mean(audio)

# Duration and Nyquist frequency describe the supplied MP3 as played.
duration = len(audio) / fs
nyquist = fs / 2

# ONC sped the audio up 10×. Use this factor only to *estimate*
# the original time and frequency scales; it does not change the MP3 file.
speedup_factor = 10

print(f"MP3 playback duration: {duration:.2f} s")
print(f"Estimated original-duration equivalent: {duration * speedup_factor:.1f} s")
print(f"Playback sampling frequency: {fs:,} Hz")
print(f"Playback Nyquist frequency: {nyquist:,.0f} Hz")
print("Note: these are not calibrated underwater sound-pressure measurements.")

### Listen before analysing

Play the supplied recording. What do you hear? Are there repeated calls, tones or changes in pitch? Write down two observations before looking at the graphs.

**Listening note:** ONC sped the recording up 10×. This playback is not what the whale calls would sound like at their original underwater frequencies. Sound playback is optional: the graphs and explanations also allow students to complete the analysis without listening.

In [ ]:
# Play the supplied, already-accelerated MP3 inside Jupyter.
# This cell does not modify the audio used in the analyses.
display(Audio(filename=str(file_path)))

## 1. Inspect the waveform

A **waveform** shows how a signal's amplitude changes through time. Its horizontal axis is *playback time (s)*, and its vertical axis is *relative digital amplitude* (not calibrated pressure).

Look for changes in amplitude, repeated patterns, quieter intervals and stronger events. We will use fewer points **for the overview plot only** so that it draws quickly; all samples are retained for analysis.

### Questions

1. Does the signal contain separate events or a single continuous tone?
2. Is its amplitude constant through time?
3. Can you identify repeated patterns?

In [ ]:
# Each recorded sample has a position in playback time (seconds).
time = np.arange(len(audio)) / fs

# Display at most ~40,000 evenly spaced samples for a fast overview.
# This changes only the drawing, NOT the audio or scientific calculations.
plot_step = max(1, len(audio) // 40000)

plt.figure(figsize=(12, 4))
plt.plot(time[::plot_step], audio[::plot_step], lw=0.7)
plt.xlabel("Playback time (s)")
plt.ylabel("Relative amplitude (digital units)")
plt.title("Blue Whale in Cascadia Basin — Waveform (10× accelerated)")
plt.tight_layout()
plt.show()

# Important: a simplified overview may miss very short peaks.
# Use a zoomed segment and the full-resolution data for detailed analysis.

In [ ]:
# Try changing these two values to inspect another playback interval.
start_time = 0
end_time = min(10, duration)

# Convert seconds into sample indices.
start_index = int(start_time * fs)
end_index = min(int(end_time * fs), len(audio))

if start_index < 0 or end_index <= start_index:
    raise ValueError("Choose times with 0 <= start_time < end_time.")

# Reduce points in the display only, for a responsive figure.
segment_step = max(1, (end_index - start_index) // 40000)

plt.figure(figsize=(12, 4))
plt.plot(
    time[start_index:end_index:segment_step],
    audio[start_index:end_index:segment_step],
    lw=0.7
)
plt.xlabel("Playback time (s)")
plt.ylabel("Relative amplitude (digital units)")
plt.title(f"Waveform — Playback {start_time}–{end_time:.1f} s")
plt.tight_layout()
plt.show()

### Try it yourself

Change `start_time` and `end_time` in the previous cell. Compare the **0–10 s** and **30–40 s** playback intervals.

1. Which section shows more prominent variations in amplitude?
2. Can you see repeated features? Are they identical in shape and amplitude?
3. Does a large amplitude, on its own, prove that a sound came from a whale?

**Remember:** 10 seconds of playback represents approximately 100 seconds of the original underwater recording. A waveform alone rarely identifies an acoustic source with certainty.

## 2. Frequency analysis — Fast Fourier Transform (FFT)

The waveform helps us see **when** the signal changes. The **FFT** helps us investigate **which frequencies** are present during the analysed interval.

**Time domain → FFT → Frequency domain**

An FFT shows peaks at frequencies that make important contributions to the recording. A spectrum computed from the **whole file** does not indicate exactly when those frequencies occurred; we will use a spectrogram for that.

**Units:** Because this MP3 is not calibrated to acoustic pressure, the FFT graph below shows **relative FFT magnitude**. It is neither a calibrated pressure spectrum nor an underwater sound pressure level. The frequency axis refers to the **accelerated playback**.

In [ ]:
# The FFT uses the full audio signal (not the simplified waveform plot).
N = len(audio)

# rfft calculates the FFT of a real-valued signal.
fft_values = np.fft.rfft(audio)

# The frequency values associated with the FFT output (playback Hz).
frequencies = np.fft.rfftfreq(N, d=1 / fs)

# Magnitude of the FFT, divided by the number of samples.
# This is useful for relative comparisons, NOT calibrated pressure.
fft_magnitude = np.abs(fft_values) / N

# Show a broad range without plotting millions of unnecessary points.
overview_max_frequency = 2000  # playback Hz
mask_overview = frequencies <= min(overview_max_frequency, nyquist)

plt.figure(figsize=(10, 5))
plt.plot(frequencies[mask_overview], fft_magnitude[mask_overview], lw=0.8)
plt.xlabel("Playback frequency (Hz)")
plt.ylabel("Relative FFT magnitude (digital units)")
plt.title("Frequency Spectrum — Accelerated Blue Whale Recording")
plt.tight_layout()
plt.show()

In [ ]:
# Zoom in on the lower part of the playback-frequency spectrum.
# Change this number to explore different frequency ranges.
max_frequency = 650  # Hz, in the sped-up recording
mask_zoom = frequencies <= min(max_frequency, nyquist)

plt.figure(figsize=(10, 5))
plt.plot(frequencies[mask_zoom], fft_magnitude[mask_zoom], lw=0.8)
plt.xlabel("Playback frequency (Hz)")
plt.ylabel("Relative FFT magnitude (digital units)")
plt.title(f"Frequency Spectrum — 0 to {max_frequency} Hz (playback)")
plt.tight_layout()
plt.show()

# To estimate the original underwater frequency, divide playback Hz by 10.

### FFT questions

1. At approximately which **playback frequencies** do you see prominent peaks?
2. Convert one of these frequencies into the approximate **original underwater frequency** by dividing by 10.
3. Are the spectral components all concentrated in one narrow band?
4. Change `max_frequency` in the previous cell. Does zooming help you see the peaks?
5. Can the FFT alone prove that every peak comes from a blue whale?

**Check your observations:** Strong spectral components occur around **145–160 Hz** and **450–470 Hz** in playback, corresponding to approximately **14.5–16 Hz** and **45–47 Hz** before acceleration. Other components also occur. Spectral peaks alone do not establish source identity: recordings can include background and instrument-related sounds.

## 3. Spectrogram — frequencies through time

The FFT above summarises frequencies across the complete recording but does not tell us **when** they occur. A **spectrogram** analyses many short, overlapping sections of the signal.

- **x-axis:** Playback time (s)
- **y-axis:** Playback frequency (Hz)
- **colour:** Relative spectral power (the strength of each time–frequency component)

A longer analysis window makes nearby frequencies easier to distinguish, but reduces the ability to pinpoint very rapid changes. Here we use 8,192 samples per window, equivalent to a frequency-bin spacing of approximately **5.4 Hz** at a sampling frequency of 44,100 Hz.

**Important:** We express power in **relative decibels**, setting the largest value *within the displayed frequency band* to **0 dB**. This is **not dB re 1 µPa** and cannot be interpreted as an underwater SPL measurement.

In [ ]:
# Calculate the spectrogram with overlapping windows.
# nperseg is the number of samples in each short FFT window.
# noverlap is the number of samples shared by neighbouring windows.
nperseg = 8192
noverlap = 6144
frequencies_s, times_s, Sxx = spectrogram(
    audio,
    fs=fs,
    nperseg=nperseg,
    noverlap=noverlap
)

# Focus on the lower playback frequencies for the blue whale example.
max_spectrogram_frequency = 650
frequency_mask = frequencies_s <= max_spectrogram_frequency
Sxx_visible = Sxx[frequency_mask, :]

# Convert to a *relative* decibel scale, not calibrated acoustic SPL.
# The largest displayed power value is 0 dB; weaker values are negative.
reference_power = np.max(Sxx_visible)
Sxx_relative_db = 10 * np.log10(
    np.maximum(Sxx_visible / reference_power, 1e-10)
)

plt.figure(figsize=(12, 6))
plt.pcolormesh(
    times_s,
    frequencies_s[frequency_mask],
    Sxx_relative_db,
    shading="auto",
    vmin=-70,
    vmax=0
)
plt.colorbar(label="Relative spectral power (dB; 0 = maximum shown)")
plt.xlabel("Playback time (s)")
plt.ylabel("Playback frequency (Hz)")
plt.title("Blue Whale in Cascadia Basin — Spectrogram (10× accelerated)")
plt.tight_layout()
plt.show()

print(f"Frequency-bin spacing: {fs / nperseg:.2f} Hz (playback scale)")

### Interpret the spectrogram

1. Which **playback frequency ranges** contain prominent patterns?
2. Do the acoustic patterns remain at exactly the same frequency over time?
3. Can you identify repeated calls or structures?
4. What does the spectrogram show that is difficult to identify from the waveform?
5. What does the FFT show differently from this spectrogram?

**Suggested interpretation:** Repeated acoustic structures occur in the accelerated recording, with particularly noticeable energy near **150 Hz** and **450–470 Hz** (approximately **15 Hz** and **45–47 Hz** in the original recording). These patterns change with time. The waveform emphasises timing and amplitude, the FFT summarises frequency components, and the spectrogram shows **which frequencies occur when**. These features are consistent with the supplied whale-recording context but do not independently prove that every component came from a whale.

## 4. Filtering the recording

A **band-pass filter** retains a chosen range of frequencies and reduces those outside it. It does **not** necessarily separate sound sources, because different sources can overlap in frequency.

From our FFT and spectrogram, a prominent playback-frequency component occurs around **150 Hz**. We will therefore select a **120–200 Hz band-pass filter** to examine this part of the accelerated recording more clearly. Before acceleration, that is approximately **12–20 Hz**.

This choice is **based on the supplied audio**, not on an assumed original-speed whale-call frequency. The earlier 10–100 Hz playback filter would remove most of the prominent component near 150 Hz.

**Predict:** What will happen to the stronger components near **450–470 Hz** after filtering?

Keep the original signal for comparison: filtering changes what is visible and audible, and may remove useful information.

In [ ]:
# A 4th-order Butterworth band-pass filter for the sped-up recording.
# All cutoff frequencies here are PLAYBACK Hz, not original underwater Hz.
lowcut = 120
highcut = 200

if not (0 < lowcut < highcut < nyquist):
    raise ValueError("Filter cutoffs must satisfy 0 < lowcut < highcut < Nyquist.")

# SOS = second-order sections, a numerically stable filter representation.
sos = butter(
    4,
    [lowcut, highcut],
    btype="bandpass",
    fs=fs,
    output="sos"
)

# Apply the filter forward and backward to avoid shifting features in time.
# Edge effects can occur near the beginning and end of the recording.
filtered_audio = sosfiltfilt(sos, audio)

print(f"Band-pass filter: {lowcut}–{highcut} Hz (playback)")
print(f"Approximate original-frequency equivalent: {lowcut/10:.0f}–{highcut/10:.0f} Hz")

In [ ]:
# Compare original and filtered waveforms.
# Plot fewer samples for display only; filtering used the entire signal.
plot_step = max(1, len(audio) // 40000)

plt.figure(figsize=(12, 4))
plt.plot(
    time[::plot_step], audio[::plot_step],
    label="Original (sped-up MP3)", alpha=0.45, lw=0.7
)
plt.plot(
    time[::plot_step], filtered_audio[::plot_step],
    label="Filtered (120–200 Hz)", lw=0.9
)
plt.xlabel("Playback time (s)")
plt.ylabel("Relative amplitude (digital units)")
plt.title("Original and Filtered Waveforms")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# FFT of the filtered recording, using the same scaling as the original.
fft_filtered = np.fft.rfft(filtered_audio)
filtered_fft_magnitude = np.abs(fft_filtered) / N

# Display the part of the spectrum relevant to the filter.
comparison_max_frequency = 650
mask_compare = frequencies <= comparison_max_frequency

plt.figure(figsize=(10, 5))
plt.plot(
    frequencies[mask_compare], fft_magnitude[mask_compare],
    label="Original", alpha=0.5, lw=0.9
)
plt.plot(
    frequencies[mask_compare], filtered_fft_magnitude[mask_compare],
    label="Filtered (120–200 Hz)", lw=0.9
)
plt.axvline(lowcut, color="grey", linestyle="--", alpha=0.6)
plt.axvline(highcut, color="grey", linestyle="--", alpha=0.6)
plt.xlabel("Playback frequency (Hz)")
plt.ylabel("Relative FFT magnitude (digital units)")
plt.title("Frequency Spectrum — Before and After Filtering")
plt.legend()
plt.tight_layout()
plt.show()

## Final interpretation

Compare the original and filtered waveforms and spectra.

### Questions

1. Which playback frequencies were reduced by the 120–200 Hz band-pass filter?
2. Does the component around 150 Hz become easier to investigate?
3. Was any potentially useful information removed, including energy near 450–470 Hz?
4. Which visualisation best answers each question: **when did a sound occur**, **which frequencies occurred**, and **when were those frequencies present**?
5. Why should we inspect the original recording before interpreting filtered results?
6. A playback peak occurs at 150 Hz. What is its estimated original underwater frequency?

### Check your understanding

- The filter retains most energy around **120–200 Hz playback** and attenuates frequencies outside this band. Components around **450–470 Hz** are greatly reduced.
- A clearer filtered signal does **not** prove that the retained component comes from only one source. Filtering can also remove information that matters scientifically.
- **Waveform →** timing and amplitude changes. **FFT →** frequencies across the analysed interval. **Spectrogram →** how frequencies change through time. **Filtering →** investigating selected frequency ranges.
- A frequency of **150 Hz during playback corresponds to approximately 15 Hz underwater** before the 10× acceleration.
- Because the MP3 is uncalibrated, none of our plotted amplitudes or relative dB values should be called underwater **sound pressure level (SPL)**.

### Recording provenance and reuse

**Audio:** *Blue Whale in Cascadia Basin*, Ocean Networks Canada, 18 October 2013, approximately 2,662 m depth; sped up 10× for listening.

Source: https://ibase.oceannetworks.ca/view-item?i=9838

**Educational use and redistribution:** The notebook's explanations and code can be shared under your teaching-material licence; the ONC audio remains a separate third-party asset subject to its own reuse terms. Confirm the applicable permission before redistributing the MP3 in a public CC BY 4.0 package.